In [1]:
import torch
from torch import nn

# 1. Configuration

In [2]:
cfgs = { 
    "A": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "B": [64, 64, "M", 128, 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "D": [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"],
    "E": [64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M", 512, 512, 512, 512, "M", 512, 512, 512, 512, "M"] 
    }

# 2. Architecture

In [24]:
class VGG(nn.Module):
    def __init__(self, cfg, batch_norm = False, num_classes = 1000,
                init_weights = True, drop_p = 0.5):
        super().__init__()

        self.features = self.make_layers(cfg, batch_norm)
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7)) # input image size와 상관없이 7 * 7이 되도록 avg pooling
        self.classifier = nn.Sequential(nn.Linear(512 * 7 * 7, 4096),
                                        nn.ReLU(),
                                        nn.Dropout(p = drop_p),
                                        nn.Linear(4096, 4096),
                                        nn.ReLU(),
                                        nn.Dropout(p = drop_p),
                                        nn.Linear(4096, num_classes))

        if init_weights:
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode = "fan_out", nonlinearity = "relu")
                    # mode = "fan_out" -> backward gradient 분산 유지를 위해 fan_out 기준으로 std 계산
                    # nonlinearity = "relu" -> relu가 반 제거했으므로 분산도 반으로 줄여 gain=√2로 분산을 보정
                    # 즉 사용한 분산 값은 2 / fan_out
                    if m.bias is not None:
                        nn.init.constant(m.bias, 0)
                elif isinstance(m, nn.Linear):
                    nn.init.normal_(m.weight, mean = 0, std = 0.01)
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

    def make_layers(self, cfg, batch_norm = False):
        layers = []
        in_channels = 3
        for v in cfg:
            if type(v) == int:
                if batch_norm:
                    layers += [nn.Conv2d(in_channels, v, padding = 1, bias = False),
                    nn.BatchNorm2d(v),
                    nn.ReLU()]
                else:
                    layers += [nn.Conv2d(in_channels, v, 3, padding = 1),
                               nn.ReLU()]
                in_channels = v
            else:
                layers += [nn.MaxPool2d(2)]
        return nn.Sequential(*layers)
        

## 2-1. AdaptiveAvgPooling

In [ ]:
avgpool = nn.AdaptiveAvgPool2d((4, 4)) # 행, 열 사이즈를 무조건 튜플 안의 값으로 맞춰줌
print(f"Before: {torch.randn(2, 3, 32, 32).shape}")
print(f"After: {avgpool(torch.randn(2, 3, 32, 32)).shape}")
x = torch.randn(2, 3, 2, 2)
print(f"Before: {x.shape}") 
print(f"After: {avgpool(x).shape}") # 원래보다 작은 놈이 들어오면 늘려서라도 맞춰줌


Before: torch.Size([2, 3, 32, 32])
After: torch.Size([2, 3, 4, 4])
Before: torch.Size([2, 3, 2, 2])
After: torch.Size([2, 3, 4, 4])


In [6]:
print(x)

tensor([[[[ 0.7829,  0.8140],
          [-0.3474, -0.1698]],

         [[ 2.5252,  0.1314],
          [-0.4385,  0.2841]],

         [[ 0.5218, -0.0940],
          [-0.5779,  0.0708]]],


        [[[-0.3068,  0.1173],
          [ 0.2545, -0.1443]],

         [[ 0.4485, -0.2030],
          [-0.3857,  0.2795]],

         [[ 0.5140, -0.5957],
          [ 0.9041,  0.1751]]]])


In [7]:
print(avgpool(x))

tensor([[[[ 0.7829,  0.7829,  0.8140,  0.8140],
          [ 0.7829,  0.7829,  0.8140,  0.8140],
          [-0.3474, -0.3474, -0.1698, -0.1698],
          [-0.3474, -0.3474, -0.1698, -0.1698]],

         [[ 2.5252,  2.5252,  0.1314,  0.1314],
          [ 2.5252,  2.5252,  0.1314,  0.1314],
          [-0.4385, -0.4385,  0.2841,  0.2841],
          [-0.4385, -0.4385,  0.2841,  0.2841]],

         [[ 0.5218,  0.5218, -0.0940, -0.0940],
          [ 0.5218,  0.5218, -0.0940, -0.0940],
          [-0.5779, -0.5779,  0.0708,  0.0708],
          [-0.5779, -0.5779,  0.0708,  0.0708]]],


        [[[-0.3068, -0.3068,  0.1173,  0.1173],
          [-0.3068, -0.3068,  0.1173,  0.1173],
          [ 0.2545,  0.2545, -0.1443, -0.1443],
          [ 0.2545,  0.2545, -0.1443, -0.1443]],

         [[ 0.4485,  0.4485, -0.2030, -0.2030],
          [ 0.4485,  0.4485, -0.2030, -0.2030],
          [-0.3857, -0.3857,  0.2795,  0.2795],
          [-0.3857, -0.3857,  0.2795,  0.2795]],

         [[ 0.5140,  0.5140,

## 2-2. Modules Check

In [10]:
model = nn.Sequential(
    nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 3),
    nn.ReLU(),
    nn.Sequential(
        nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3),
        nn.ReLU(),
        nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = 3),
        nn.ReLU(),
    nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = 3)
    )
)

[m for m in model.modules()]

[Sequential(
   (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
   (1): ReLU()
   (2): Sequential(
     (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
     (1): ReLU()
     (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
     (3): ReLU()
     (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
   )
 ),
 Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1)),
 ReLU(),
 Sequential(
   (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
   (1): ReLU()
   (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
   (3): ReLU()
   (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
 ),
 Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1)),
 ReLU(),
 Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1)),
 ReLU(),
 Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))]

## 2-3. nn.Sequential

In [13]:
# nn.Sequential은 리스트 형태로 넣으면 에러
# model1 = nn.Sequential([
#     nn.Linear(1, 1),
#     nn.Linear(1, 1)
# ])

# 따라서 풀어서 넣어줘야 함
model2 = nn.Sequential(nn.Linear(1, 1), 
                       nn.Linear(1, 1))

# *는 리스트 언팩킹
print([1,2])
print(*[1,2])

[1, 2]
1 2


In [ ]:
# * 앞에 넣으면 리스트 언팩킹 하기 때문에 에러 아님.
model3 = nn.Sequential(*[nn.Linear(1, 1), 
                       nn.Linear(1, 1)])

# 3. Define Model

In [26]:
model = VGG(cfgs["E"])

!pip install torchinfo
from torchinfo import summary
summary(model, input_size = (2, 3, 224, 224), device = "cpu")

/var/folders/n0/8pk3htx55dq8_0xxdgv4jl8w0000gn/T/ipykernel_50913/1055900916.py:24: FutureWarning: `nn.init.constant` is now deprecated in favor of `nn.init.constant_`.
  nn.init.constant(m.bias, 0)


Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      --                        --
├─Sequential: 1-1                        [2, 512, 7, 7]            --
│    └─Conv2d: 2-1                       [2, 64, 224, 224]         1,792
│    └─ReLU: 2-2                         [2, 64, 224, 224]         --
│    └─Conv2d: 2-3                       [2, 64, 224, 224]         36,928
│    └─ReLU: 2-4                         [2, 64, 224, 224]         --
│    └─MaxPool2d: 2-5                    [2, 64, 112, 112]         --
│    └─Conv2d: 2-6                       [2, 128, 112, 112]        73,856
│    └─ReLU: 2-7                         [2, 128, 112, 112]        --
│    └─Conv2d: 2-8                       [2, 128, 112, 112]        147,584
│    └─ReLU: 2-9                         [2, 128, 112, 112]        --
│    └─MaxPool2d: 2-10                   [2, 128, 56, 56]          --
│    └─Conv2d: 2-11                      [2, 256, 56, 56]          29